Conectado a Python 3.14.2

In [ ]:
import pandas as pd
import datetime


#importando arquivo csv

In [ ]:
df = pd.read_csv('acidentes-GO-2024_2025_limpo.csv')
df['data_inversa'] = pd.to_datetime(df['data_inversa'])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6501 entries, 0 to 6500
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id                      6501 non-null   int64         
 1   data_inversa            6501 non-null   datetime64[ns]
 2   dia_semana              6501 non-null   object        
 3   horario                 6501 non-null   object        
 4   uf                      6501 non-null   object        
 5   br                      6501 non-null   int64         
 6   km                      6501 non-null   float64       
 7   municipio               6501 non-null   object        
 8   causa_acidente          6501 non-null   object        
 9   tipo_acidente           6501 non-null   object        
 10  classificacao_acidente  6501 non-null   object        
 11  fase_dia                6501 non-null   object        
 12  sentido_via             6501 non-null   object  

In [ ]:
# 1. Converte a coluna 'horario' para extrair a hora (0 a 23)
hora_num = pd.to_datetime(df['horario'], format='%H:%M:%S').dt.hour

# 2. Cria a regra para categorizar os 4 turnos
def classificar_turno(h):
    if 6 <= h < 12:
        return '1. Manhã (06h-11h59)'
    elif 12 <= h < 18:
        return '2. Tarde (12h-17h59)'
    elif 18 <= h < 24:
        return '3. Noite (18h-23h59)'
    else:
        return '4. Madrugada (00h-05h59)'

# 3. Aplica a função para criar a coluna 'turno'
df['turno'] = hora_num.apply(classificar_turno)

# 4. Gera a tabela cruzada simples: Turno x Sentido da Via
tabela_turno_sentido = pd.crosstab(
    df['turno'], 
    df['sentido_via'], 
    margins=True, 
    margins_name='TOTAL'
)

print("=== QUANTIDADE DE ACIDENTES POR TURNO E SENTIDO DA VIA ===")
print(tabela_turno_sentido)

=== QUANTIDADE DE ACIDENTES POR TURNO E SENTIDO DA VIA ===
sentido_via               Crescente  Decrescente  Não Informado  TOTAL
turno                                                                 
1. Manhã (06h-11h59)            928          786              3   1717
2. Tarde (12h-17h59)           1103          885              6   1994
3. Noite (18h-23h59)           1039          820              6   1865
4. Madrugada (00h-05h59)        511          413              1    925
TOTAL                          3581         2904             16   6501


In [ ]:
# 1. Extrai a hora exata (0 a 23) da coluna 'horario'
df['hora_exata'] = pd.to_datetime(df['horario'], format='%H:%M:%S').dt.hour

# 2. Conta a quantidade de acidentes para cada hora
acidentes_por_hora = df['hora_exata'].value_counts().sort_index()

# 3. Exibe o resultado de 0h até 23h
print("=== ACIDENTES POR HORA EXATA (0h as 23h) ===")
for hora, total in acidentes_por_hora.items():
    print(f"{hora:02d}h: {total} acidentes")

=== ACIDENTES POR HORA EXATA (0h as 23h) ===
00h: 146 acidentes
01h: 134 acidentes
02h: 120 acidentes
03h: 137 acidentes
04h: 144 acidentes
05h: 244 acidentes
06h: 287 acidentes
07h: 377 acidentes
08h: 293 acidentes
09h: 252 acidentes
10h: 248 acidentes
11h: 260 acidentes
12h: 281 acidentes
13h: 278 acidentes
14h: 319 acidentes
15h: 344 acidentes
16h: 373 acidentes
17h: 399 acidentes
18h: 426 acidentes
19h: 454 acidentes
20h: 333 acidentes
21h: 257 acidentes
22h: 212 acidentes
23h: 183 acidentes


In [ ]:
# 1. Extrai a hora exata
df['hora_exata'] = pd.to_datetime(df['horario'], format='%H:%M:%S').dt.hour

# 2. Cria a tabela cruzada de Hora x Sentido da Via
tabela_hora_sentido = pd.crosstab(
    df['hora_exata'], 
    df['sentido_via']
)

# 3. Filtra a janela crítica da tarde/noite (16h às 21h) para comparar os sentidos
janela_pico = tabela_hora_sentido.loc[16:21, ['Crescente', 'Decrescente']]

# Adiciona o percentual do sentido Crescente no total daquela hora
janela_pico['Total_Hora'] = janela_pico['Crescente'] + janela_pico['Decrescente']
janela_pico['%_Crescente'] = (janela_pico['Crescente'] / janela_pico['Total_Hora'] * 100).round(1)

print("=== JANELA CRÍTICA (16h as 21h) - CRESCENTE vs DECRESCENTE ===")
print(janela_pico)

=== JANELA CRÍTICA (16h as 21h) - CRESCENTE vs DECRESCENTE ===
sentido_via  Crescente  Decrescente  Total_Hora  %_Crescente
hora_exata                                                  
16                 209          163         372         56.2
17                 223          176         399         55.9
18                 239          184         423         56.5
19                 260          193         453         57.4
20                 180          153         333         54.1
21                 135          121         256         52.7


In [ ]:
# Tabela cruzada entre Fase do Dia e Sentido da Via
tabela_fase = pd.crosstab(
    df['fase_dia'], 
    df['sentido_via'], 
    margins=True, 
    margins_name='Total'
)

# Cálculo do percentual no sentido Crescente
tabela_fase['%_Crescente'] = (tabela_fase['Crescente'] / tabela_fase['Total'] * 100).round(1)

# Ordenação pelo total de ocorrências
tabela_fase_ordenada = tabela_fase.sort_values(by='Total', ascending=False)

# Exibição limpa sem índices desalinhados
print("=== ACIDENTES POR FASE DO DIA x SENTIDO DA VIA ===")
print(tabela_fase_ordenada.to_string())

=== ACIDENTES POR FASE DO DIA x SENTIDO DA VIA ===
sentido_via  Crescente  Decrescente  Não Informado  Total  %_Crescente
fase_dia                                                              
Total             3581         2904             16   6501         55.1
Pleno dia         1882         1590             10   3482         54.0
Plena Noite       1276         1020              4   2300         55.5
Amanhecer          208          162              0    370         56.2
Anoitecer          215          132              2    349         61.6


In [ ]:
# Contagem total de ocorrências por sentido da via
contagem_sentido = df['sentido_via'].value_counts()

# Exibe o resultado com a porcentagem
porcentagem_sentido = df['sentido_via'].value_counts(normalize=True) * 100

# Tabela consolidada
resumo_sentido = pd.DataFrame({
    'Total_Acidentes': contagem_sentido,
    'Porcentagem (%)': porcentagem_sentido.round(2)
})

print("=== CONTAGEM TOTAL POR SENTIDO DA VIA ===")
print(resumo_sentido.to_string())

=== CONTAGEM TOTAL POR SENTIDO DA VIA ===
               Total_Acidentes  Porcentagem (%)
sentido_via                                    
Crescente                 3581            55.08
Decrescente               2904            44.67
Não Informado               16             0.25


In [ ]:
# 1. Cria a coluna com a hora exata
df['hora_num'] = pd.to_datetime(df['horario'], format='mixed', errors='coerce').dt.hour

# 2. Categoriza as horas em períodos do dia
df['periodo_dia'] = pd.cut(
    df['hora_num'],
    bins=[-1, 5, 11, 17, 23],
    labels=['Madrugada (00h-05h)', 'Manhã (06h-11h)', 'Tarde (12h-17h)', 'Noite (18h-23h)']
)

# 3. Filtra as Top 5 causas
top_causas = df['causa_acidente'].value_counts().head(5).index
df_top = df[df['causa_acidente'].isin(top_causas)]

# 4. Gera a tabela agrupada por período e aplica destaque visual de cor
tabela_periodos = pd.crosstab(
    df_top['causa_acidente'], 
    df_top['periodo_dia'], 
    margins=True, 
    margins_name='Total'
)

# Renderiza com destaque de intensidade (gradiente) na Janela Interativa
tabela_periodos.style.background_gradient(cmap='YlOrRd', axis=None)

periodo_dia,Madrugada (00h-05h),Manhã (06h-11h),Tarde (12h-17h),Noite (18h-23h),Total
causa_acidente,,,,,
Acessar a via sem observar a presença dos outros veículos,28,172,181,138,519
Ausência de reação do condutor,166,237,272,275,950
Ingestão de álcool pelo condutor,139,58,119,233,549
Reação tardia ou ineficiente do condutor,142,289,291,273,995
Velocidade Incompatível,45,101,137,107,390
Total,520,857,1000,1026,3403


In [ ]:
df['classificacao_acidente'].value_counts()

classificacao_acidente
Com Vítimas Feridas    4718
Sem Vítimas            1286
Com Vítimas Fatais      497
Name: count, dtype: int64

In [ ]:
df['tipo_acidente'].value_counts()

tipo_acidente
Saída de leito carroçável         1477
Colisão traseira                  1345
Colisão transversal                591
Colisão lateral mesmo sentido      551
Tombamento                         442
Colisão com objeto                 437
Queda de ocupante de veículo       338
Atropelamento de Pedestre          281
Colisão frontal                    280
Incêndio                           204
Capotamento                        164
Colisão lateral sentido oposto     155
Atropelamento de Animal            122
Engavetamento                       75
Eventos atípicos                    30
Derramamento de carga                8
Sinistro pessoal de trânsito         1
Name: count, dtype: int64

In [ ]:
df['veiculos'].value_counts()

veiculos
2     2652
1     2568
3      731
4      321
5      131
6       49
7       21
8       16
15       3
9        3
13       2
11       1
10       1
12       1
82       1
Name: count, dtype: int64

In [ ]:
# Agrupa somando os acidentes por Dia da Semana e Sentido da Via
tabela_dias = df.groupby(['dia_semana', 'sentido_via']).size().reset_index(name='total_acidentes')

# Ordena os dias da semana em ordem lógica (de segunda a domingo)
ordem_dias = [
    'segunda-feira', 'terça-feira', 'quarta-feira', 
    'quinta-feira', 'sexta-feira', 'sábado', 'domingo'
]

# Aplica a ordenação correta
tabela_dias['dia_semana'] = pd.Categorical(tabela_dias['dia_semana'], categories=ordem_dias, ordered=True)
tabela_dias = tabela_dias.sort_values(by=['dia_semana', 'sentido_via'])

# Exibe a tabela completa
print(tabela_dias.to_string(index=False))

   dia_semana   sentido_via  total_acidentes
segunda-feira     Crescente              464
segunda-feira   Decrescente              380
segunda-feira Não Informado                1
  terça-feira     Crescente              425
  terça-feira   Decrescente              373
  terça-feira Não Informado                4
 quarta-feira     Crescente              484
 quarta-feira   Decrescente              357
 quarta-feira Não Informado                3
 quinta-feira     Crescente              484
 quinta-feira   Decrescente              358
 quinta-feira Não Informado                7
  sexta-feira     Crescente              487
  sexta-feira   Decrescente              441
  sexta-feira Não Informado                1
       sábado     Crescente              572
       sábado   Decrescente              493
      domingo     Crescente              665
      domingo   Decrescente              502


In [ ]:
# 1. Filtra apenas os dias úteis (segunda a sexta)
dias_uteis = ['segunda-feira', 'terça-feira', 'quarta-feira', 'quinta-feira', 'sexta-feira']
df_uteis = df[df['dia_semana'].isin(dias_uteis)].copy()

# 2. Extrai a hora em formato numérico (0 a 23)
df_uteis['hora'] = pd.to_datetime(df_uteis['horario'].astype(str), format='%H:%M:%S', errors='coerce').dt.hour

# 3. Filtra apenas os sentidos conhecidos
df_uteis = df_uteis[df_uteis['sentido_via'].isin(['Crescente', 'Decrescente'])]

# 4. Tabela cruzada: Linhas = Hora (0 a 23), Colunas = Sentido da Via
tabela_fluxo_horario = pd.pivot_table(
    df_uteis,
    index='hora',
    columns='sentido_via',
    values='id',
    aggfunc='count',
    fill_value=0
)

# 5. Cria uma coluna mostrando a diferença total e qual sentido liderou naquela hora
tabela_fluxo_horario['Total'] = tabela_fluxo_horario['Crescente'] + tabela_fluxo_horario['Decrescente']
tabela_fluxo_horario['Diferença (Cres - Dec)'] = tabela_fluxo_horario['Crescente'] - tabela_fluxo_horario['Decrescente']

# Exibe a tabela organizada de 0h a 23h
print("=== DISTRIBUIÇÃO DE ACIDENTES POR HORA E SENTIDO (DIAS ÚTEIS) ===")
print(tabela_fluxo_horario)

=== DISTRIBUIÇÃO DE ACIDENTES POR HORA E SENTIDO (DIAS ÚTEIS) ===
sentido_via  Crescente  Decrescente  Total  Diferença (Cres - Dec)
hora                                                              
0                   41           44     85                      -3
1                   42           30     72                      12
2                   29           29     58                       0
3                   43           34     77                       9
4                   39           30     69                       9
5                   78           75    153                       3
6                  102          104    206                      -2
7                  159          127    286                      32
8                  128          102    230                      26
9                   74           82    156                      -8
10                  86           78    164                       8
11                 100           71    171                     

Nenhum kernel conectado